# Documentación del Proceso ETL: Consolidación Histórica y Enriquecimiento Espacial del Valor de Referencia

## 1. Fuentes de Datos e Ingesta

El proyecto se alimenta de tres fuentes oficiales abiertas provistas por el Distrito Capital a través del portal de Datos Abiertos Bogotá e IDECA:

Valor de Referencia por Metro Cuadrado de Terreno: Conjunto de datos históricos en formato comprimido (.zip) que agrupa los polígonos de manzanas catastrales y sus respectivos valores económicos de referencia calculados año tras año (2012–2026).

Fuente: https://datosabiertos.bogota.gov.co/dataset/valor-de-referencia-por-metro-cuadrado-de-terreno

Sector Catastral: Capa geográfica oficial (.zip / Shapefile) que define la división político-administrativa a nivel de barrios o sectores catastrales, utilizada para contextualizar espacialmente cada manzana.

Fuente: https://datosabiertos.bogota.gov.co/dataset/sector-catastral

Localidad (loca.gpkg): Capa geoespacial en formato GeoPackage que delimita las localidades del Distrito Capital, permitiendo establecer la jerarquía territorial superior.

Fuente: https://datosabiertos.bogota.gov.co/dataset/localidad-bogota-d-c

## 2. Metodología y Procesamiento (ETL Pipeline)

El pipeline automatizado implementa un proceso de Extracción, Transformación y Carga (ETL) espacial estructurado en los siguientes pasos:

Ingestión y Estandarización: Se descomprimen de forma masiva los archivos históricos anuales, extrayendo las geometrías de las manzanas y normalizando los nombres de columnas heterogéneas (mancodigo, v_ref, shape_area, shape_len, ano) para unificarlas en un único marco de datos temporal.

Enriquecimiento Espacial por Intersección (sjoin): Utilizando la librería geopandas, se reproyectan las capas bajo un mismo sistema de referencia de coordenadas (CRS) y se ejecutan cruces espaciales secuenciales basados en el predicado intersects. Esto permite asignar a cada manzana histórica su respectivo barrio (desde el Sector Catastral) y localidad (desde el archivo GeoPackage).

Deduplicación e Integridad Estructural: Para corregir la multiplicación de registros derivada de manzanas que intersectan bordes administrativos, se implementa una limpieza basada en el identificador único original (id_original), garantizando una única asignación espacial por manzana.

Depuración de Datos Incompletos: Se aplica un filtro de exclusión (dropna) para remover registros que carezcan de variables económicas o administrativas esenciales (barrio, localidad, v_ref, shape_area, shape_len).

## 3. Productos Finales Generados

Como resultado del procesamiento, se obtienen dos datasets estructurados:

valor_referencia_historico_tabular.csv: Archivo tabular optimizado para análisis estadísticos y econométricos. Contiene el histórico consolidado a nivel de manzana, con sus valores de referencia, dimensiones físicas y jerarquía administrativa (barrio y localidad).

valor_referencia_historico_espacial.gpkg: Archivo GeoPackage que conserva la geometría vectorial completa, permitiendo visualizaciones geográficas y análisis de autocorrelación espacial en software SIG (QGIS, ArcGIS) o bibliotecas de Python.

# Librerias

In [2]:
import os
import zipfile
import glob
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from osgeo import gdal

gdal.SetConfigOption("SHAPE_RESTORE_SHX", "YES")


# 0. Extracción de datos

In [3]:
# ==============================================================================
# PASO 0: EXTRACCIÓN AUTOMÁTICA DE ARCHIVOS COMPRIMIDOS
# ==============================================================================
os.makedirs('data_historica', exist_ok=True)
for f in glob.glob('*.zip'):
    if 'sector' not in f.lower() and 'transmilenio' not in f.lower():
        with zipfile.ZipFile(f, 'r') as z:
            z.extractall(os.path.join('data_historica', os.path.splitext(f)[0]))

os.makedirs('temp_sector', exist_ok=True)
for f in glob.glob('*sector*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('temp_sector')

os.makedirs('temp_transmilenio', exist_ok=True)
for f in glob.glob('*transmilenio*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('temp_transmilenio')

print("--- EXTRACCIÓN COMPLETADA ---")

--- EXTRACCIÓN COMPLETADA ---


# 1. Unificación Localidad - Barrio - Geometria

In [4]:
# ==============================================================================
# PASO 1: DICCIONARIO ESPACIAL BASE (LOCALIDAD - BARRIO - GEOMETRÍA)
# ==============================================================================
archivo_sector = glob.glob('temp_sector/**/*.shp', recursive=True)[0]
gdf_sector = gpd.read_file(archivo_sector)

gdf_localidad = gpd.read_file('loca.gpkg')
gdf_localidad = gdf_localidad.to_crs(gdf_sector.crs)

gdf_sector = gdf_sector.rename(columns={'SCANOMBRE': 'barrio'})

gdf_barrios_base = gpd.sjoin(gdf_sector[['barrio', 'geometry']], gdf_localidad[['LocNombre', 'geometry']], how='left', predicate='intersects')
gdf_barrios_base = gdf_barrios_base.rename(columns={'LocNombre': 'localidad'})
gdf_barrios_base = gdf_barrios_base.drop(columns=['index_right'])

# Deduplicar polígonos de barrio
gdf_barrios_base = gdf_barrios_base.drop_duplicates(subset=['barrio']).reset_index(drop=True)

print(f"\n[PASO 1] Tamaño del Diccionario Espacial Base (Barrios): {gdf_barrios_base.shape[0]} filas, {gdf_barrios_base.shape[1]} columnas")


[PASO 1] Tamaño del Diccionario Espacial Base (Barrios): 1150 filas, 3 columnas


In [5]:
gdf_barrios_base.head(5)

,barrio,geometry,localidad
0,EL BOSQUE,"POLYGON ((-74.10015 4.50545, -74.10016 4.50525...",USME
1,BILBAO,"POLYGON ((-74.11315 4.75187, -74.11371 4.751, ...",SUBA
2,CIUDAD KENNEDY CENTRAL,"POLYGON ((-74.15601 4.61652, -74.15601 4.61657...",KENNEDY
3,CONCEPCION NORTE,"POLYGON ((-74.06568 4.65636, -74.06511 4.65727...",CHAPINERO
4,COLON,"POLYGON ((-74.11964 4.61964, -74.11899 4.62043...",PUENTE ARANDA


# 2. Consolidación valores historicos de metro cuadrado

In [7]:
# ==============================================================================
# PASO 2: CONSOLIDACIÓN DE VALORES DE REFERENCIA HISTÓRICOS (VHI)
# ==============================================================================
lista_gdfs = []
target_crs = "EPSG:4326"
 
for shp in glob.glob('data_historica/**/*.shp', recursive=True):
    gdf = gpd.read_file(shp)
   
    # Asegurar que todas las capas usen el mismo CRS para poder concatenar
    if gdf.crs is None:
        gdf.set_crs(target_crs, inplace=True)
    elif gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)
       
    match = re.search(r'20\d{2}', shp)
    ano = int(match.group(0)) if match else None
    cols = {c.lower(): c for c in gdf.columns}
    datos = {
        'mancodigo': gdf[cols.get('mancodigo')] if cols.get('mancodigo') else None,
        'v_ref': gdf[cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')] if (cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')) else None,
        'ano': ano,
        'geometry': gdf['geometry']
    }
    lista_gdfs.append(gpd.GeoDataFrame(datos, crs=target_crs))
 
gdf_maestro = pd.concat(lista_gdfs, ignore_index=True)
print(f"[PASO 2a] Tamaño de Registros de Manzanas Históricas (gdf_maestro): {gdf_maestro.shape[0]} filas, {gdf_maestro.shape[1]} columnas")
 
gdf_barrios_base = gdf_barrios_base.to_crs(gdf_maestro.crs)
 
# Cruce espacial de manzanas con diccionario de barrios
gdf_historico = gpd.sjoin(gdf_maestro, gdf_barrios_base[['barrio', 'localidad', 'geometry']], how='inner', predicate='intersects')
if 'index_right' in gdf_historico.columns:
    gdf_historico = gdf_historico.drop(columns=['index_right'])
 
# Mediana del valor de referencia por barrio y año
df_vhi_barrio = gdf_historico.groupby(['localidad', 'barrio', 'ano'])['v_ref'].median().reset_index()
print(f"[PASO 2b] Tamaño del Consolidado VHI por Barrio y Año: {df_vhi_barrio.shape[0]} filas, {df_vhi_barrio.shape[1]} columnas")

/var/folders/63/cdh2flb14xn_3wgvc3t8vlr00000gn/T/ipykernel_15441/4293886475.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  gdf_maestro = pd.concat(lista_gdfs, ignore_index=True)


[PASO 2a] Tamaño de Registros de Manzanas Históricas (gdf_maestro): 519088 filas, 4 columnas
[PASO 2b] Tamaño del Consolidado VHI por Barrio y Año: 13147 filas, 4 columnas


In [8]:
df_vhi_barrio.head(5)

,localidad,barrio,ano,v_ref
0,ANTONIO NARIÑO,CARACAS,2015,1300000.0
1,ANTONIO NARIÑO,CARACAS,2016,1400000.0
2,ANTONIO NARIÑO,CARACAS,2017,1700000.0
3,ANTONIO NARIÑO,CARACAS,2018,1700000.0
4,ANTONIO NARIÑO,CARACAS,2019,1750000.0


# 3. Extracción datos transmilenio

In [9]:
# ==============================================================================
# PASO 3: DENSIDAD DE INFRAESTRUCTURA DE TRANSMILENIO
# ==============================================================================
archivo_transmilenio = glob.glob('temp_transmilenio/**/*.shp', recursive=True)[0]
gdf_transmilenio = gpd.read_file(archivo_transmilenio)
print(f"[PASO 3a] Estaciones de Transmilenio leídas: {gdf_transmilenio.shape[0]} filas, {gdf_transmilenio.shape[1]} columnas")

if gdf_barrios_base.crs != gdf_transmilenio.crs:
    gdf_transmilenio = gdf_transmilenio.to_crs(gdf_barrios_base.crs)

# Conteo espacial de estaciones por barrio
estaciones_por_barrio = gpd.sjoin(gdf_transmilenio, gdf_barrios_base[['barrio', 'geometry']], how='inner', predicate='intersects')
conteo_estaciones = estaciones_por_barrio.groupby('barrio').size().reset_index(name='num_estaciones_transmilenio')
print(f"[PASO 3b] Barrios con al menos una estación de Transmilenio: {conteo_estaciones.shape[0]} barrios")

[PASO 3a] Estaciones de Transmilenio leídas: 153 filas, 28 columnas
[PASO 3b] Barrios con al menos una estación de Transmilenio: 111 barrios


In [10]:
conteo_estaciones

,barrio,num_estaciones_transmilenio
0,ABRAHAM LINCOLN,1
1,ACEVEDO TEJADA,2
2,ALTOS DE CHOZICA,1
3,AUTOPISTA MUZU,1
4,AUTOPISTA MUZU ORIENTAL,1
...,...,...
106,SIMON BOLIVAR,1
107,SOCIEGO,1
108,SOSIEGO SUR,1
109,SURAMERICA,1


# 4. Cálculo CAGR 2015-2025

In [16]:
# ==============================================================================
# PASO 4: CÁLCULO DE LA TASA DE CRECIMIENTO ANUAL COMPUESTA (CAGR 2015-2025)
# ==============================================================================
df_cagr_base = df_vhi_barrio[df_vhi_barrio['ano'].isin([2015, 2025, 2026])]
df_pivot = df_cagr_base.pivot(index=['localidad', 'barrio'], columns='ano', values='v_ref').reset_index()
df_pivot = df_pivot.rename(columns={2015: 'v_ref_2015', 2025: 'v_ref_2025', 2026: 'v_ref_2026'})

# Calcular CAGR de 11 años (2015 a 2025)
n_anos = 2025 - 2015
df_pivot['cagr_15_25'] = (df_pivot['v_ref_2025'] / df_pivot['v_ref_2015']) ** (1 / n_anos) - 1
print(f"[PASO 4] Tamaño de la tabla de CAGR (2015-2025): {df_pivot.shape[0]} filas, {df_pivot.shape[1]} columnas")

[PASO 4] Tamaño de la tabla de CAGR (2015-2025): 1103 filas, 6 columnas


In [37]:
df_pivot

ano,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25
0,ANTONIO NARIÑO,CARACAS,1300000.00,2679000.0,3200000.0,0.074986
1,ANTONIO NARIÑO,EDUARDO SANTOS,900000.00,2950000.0,3000000.0,0.126051
2,ANTONIO NARIÑO,EL VERGEL,912500.00,2900000.0,3000000.0,0.122578
3,ANTONIO NARIÑO,LA FRAGUITA,1000000.00,2900000.0,3000000.0,0.112346
4,ANTONIO NARIÑO,SANTANDER,1000000.00,3040000.0,3150000.0,0.117602
...,...,...,...,...,...,...
1098,USME,VILLA ISRAEL,200000.00,800000.0,295200.0,0.148698
1099,USME,VILLABEL,135000.00,203000.0,393034.0,0.041637
1100,USME,YOMASA,230000.00,450000.0,480000.0,0.069420
1101,USME,YOMASA NORTE,34001.50,170694.0,170124.0,0.175092


# 5. Cargar listings agrupados

In [34]:
listings = pd.read_csv("listings_agrupado_barrio_localidad.csv")
len(listings)

654

In [35]:
listings = listings.rename(columns={'conteo': 'num_listings_airbnb'})

In [36]:
listings

,localidad,barrio,num_listings_airbnb
0,ANTONIO NARIÑO,CIUDAD BERNA,1
1,ANTONIO NARIÑO,CIUDAD JARDIN SUR,4
2,ANTONIO NARIÑO,EDUARDO FREY,5
3,ANTONIO NARIÑO,LA FRAGUA,1
4,ANTONIO NARIÑO,LA FRAGUITA,7
...,...,...,...
649,USME,LA FISCALA NORTE,1
650,USME,LA REGADERA,1
651,USME,MONTEBLANCO,1
652,USME,OLARTE,3


# 5. Unificación datos

In [41]:
# ==============================================================================
# PASO 5: UNIFICACIÓN TOTAL, ORDENAMIENTO Y EXPORTACIÓN
# ==============================================================================
matriz_final = gdf_barrios_base.copy()

# Fusión con CAGR y valores de referencia
# CORRECCIÓN: Se agregó 'localidad' a la lista de columnas seleccionadas de df_pivot
matriz_final = matriz_final.merge(df_pivot[['barrio', 'localidad', 'v_ref_2015', 'v_ref_2025', 'v_ref_2026', 'cagr_15_25']], on=['barrio', 'localidad'], how='left')

# Fusión con Transmilenio
matriz_final = matriz_final.merge(conteo_estaciones, on=['barrio'], how='left')
matriz_final['num_estaciones_transmilenio'] = matriz_final['num_estaciones_transmilenio'].fillna(0).astype(int)

# Cruzamos la información manteniendo todos los barrios del dataset general
matriz_final = matriz_final.merge(listings, on=['barrio', 'localidad'], how='left')

# Rellenamos con 0 los valores de los barrios que no estaban en listings
matriz_final['num_listings_airbnb'] = matriz_final['num_listings_airbnb'].fillna(0)

# Manejo de nulos en CAGR si un barrio no tiene serie completa
matriz_final['cagr_15_25'] = matriz_final['cagr_15_25'].fillna(0)

# ---> ORDENAR LAS COLUMNAS SEGÚN LO SOLICITADO <---
columnas_ordenadas = [
    'localidad',
    'barrio',
    'v_ref_2015',
    'v_ref_2025',
    'v_ref_2026',
    'cagr_15_25',
    'num_estaciones_transmilenio',
    'num_listings_airbnb'
] + ['geometry']

matriz_final = matriz_final[columnas_ordenadas]

# Asegurar que se mantenga como GeoDataFrame espacial
matriz_final = gpd.GeoDataFrame(matriz_final, geometry='geometry')

# Exportación de archivos finales ordenados
matriz_final.to_csv('matriz_barrios_unificada.csv', index=False)
matriz_final.to_file('matriz_barrios_unificada.gpkg', driver='GPKG')

print("\n¡Matriz unificada, ordenada y exportada con éxito!")
print(f"Columnas finales actuales: {matriz_final.columns.tolist()}")


¡Matriz unificada, ordenada y exportada con éxito!
Columnas finales actuales: ['localidad', 'barrio', 'v_ref_2015', 'v_ref_2025', 'v_ref_2026', 'cagr_15_25', 'num_estaciones_transmilenio', 'num_listings_airbnb', 'geometry']


In [42]:
matriz_final

,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,geometry
0,USME,EL BOSQUE,290000.0,650000.0,660000.0,0.084056,0,0.0,"POLYGON ((-74.10015 4.50545, -74.10016 4.50525..."
1,SUBA,BILBAO,750000.0,1790000.0,2050000.0,0.090886,0,0.0,"POLYGON ((-74.11315 4.75187, -74.11371 4.751, ..."
2,KENNEDY,CIUDAD KENNEDY CENTRAL,1000000.0,2900000.0,2640000.0,0.112346,0,7.0,"POLYGON ((-74.15601 4.61652, -74.15601 4.61657..."
3,CHAPINERO,CONCEPCION NORTE,1400000.0,3580000.0,4380000.0,0.098438,1,0.0,"POLYGON ((-74.06568 4.65636, -74.06511 4.65727..."
4,PUENTE ARANDA,COLON,1300000.0,2850000.0,3170000.0,0.081658,0,16.0,"POLYGON ((-74.11964 4.61964, -74.11899 4.62043..."
...,...,...,...,...,...,...,...,...,...
1145,CIUDAD BOLIVAR,MEISSEN,64000.0,1198158.0,1395000.0,0.340397,0,1.0,"POLYGON ((-74.13463 4.56098, -74.13427 4.56086..."
1146,RAFAEL URIBE URIBE,LOS MOLINOS,450000.0,2750000.0,2600000.0,0.198428,0,0.0,"POLYGON ((-74.10753 4.55803, -74.10742 4.55802..."
1147,SUBA,EL PLAN,1400000.0,3400000.0,3798483.0,0.092786,0,38.0,"POLYGON ((-74.06395 4.74181, -74.06391 4.74181..."
1148,CIUDAD BOLIVAR,ISMAEL PERDOMO,400000.0,1040000.0,1050000.0,0.100265,0,0.0,"POLYGON ((-74.16717 4.58155, -74.16718 4.58155..."
